# BTL02 – Chạy thực nghiệm trên Colab / Kaggle (GPU)

Phụ trách: Huỳnh Ngọc Trúc

Notebook này chỉ gọi lại đúng các lệnh chạy trên máy (`python -m src.train`, `python -m experiments.run_E*`), không chứa code mô hình riêng.

**Trước khi mở notebook:** commit và push code mới nhất lên GitHub, vì notebook `git clone` repo.

| | Colab | Kaggle |
|---|---|---|
| GPU | Runtime → Change runtime type → T4 GPU | Settings → Accelerator → GPU T4 x2 hoặc P100; bật Internet |
| Dữ liệu | Đặt `fer2013.csv` ở `MyDrive/fer2013.csv` | Add Input → dataset `nicolejyt/facialexpressionrecognition` |
| Mất kết nối | `results/` được nối vào `MyDrive/BTL02_results`, chạy lại ô là tiếp tục run còn thiếu | Dùng **Save Version → Save & Run All** để chạy nền (tối đa 12 giờ), kết quả ở tab Output |

Chạy xong: tải file `results_*.zip`, giải nén đè vào thư mục `BTL02 Emotion Recognition/` trong repo, rồi ghi nhật ký từng run (`/btl02-ghi-run <mã run>`), cột thời gian ghi kèm loại GPU.

In [ ]:
!nvidia-smi

In [ ]:
import glob, os, shutil, subprocess, time

KAGGLE = os.path.exists("/kaggle")
COLAB = not KAGGLE and os.path.exists("/content")
WORK = "/kaggle/working" if KAGGLE else "/content"
REPO = os.path.join(WORK, "deep-learning-assignments")
URL = "https://github.com/TysonNgo25/deep-learning-assignments.git"

if os.path.exists(REPO):
    subprocess.run(["git", "-C", REPO, "pull", "--ff-only"], check=True)
else:
    subprocess.run(["git", "clone", "--depth", "1", URL, REPO], check=True)
os.chdir(os.path.join(REPO, "BTL02 Emotion Recognition"))
print("Thư mục làm việc:", os.getcwd())
print(subprocess.run(["git", "log", "-1", "--format=%h %ad %s", "--date=short"],
                     capture_output=True, text=True).stdout)

In [ ]:
# Dữ liệu: Kaggle lấy từ Input, Colab lấy từ Google Drive.
if COLAB:
    from google.colab import drive
    drive.mount("/content/drive")

os.makedirs("data", exist_ok=True)
if not os.path.exists("data/fer2013.csv"):
    found = glob.glob("/kaggle/input/**/fer2013.csv", recursive=True) +             glob.glob("/content/drive/MyDrive/fer2013.csv")
    if not found:
        raise FileNotFoundError("Không thấy fer2013.csv (Kaggle: Add Input; Colab: MyDrive/fer2013.csv)")
    shutil.copy(found[0], "data/fer2013.csv")
print("data/fer2013.csv:", os.path.getsize("data/fer2013.csv") // 2**20, "MB")

# Colab: nối results/ vào Drive để không mất log khi bị ngắt kết nối.
if COLAB and not os.path.islink("results"):
    drive_results = "/content/drive/MyDrive/BTL02_results"
    os.makedirs(drive_results, exist_ok=True)
    shutil.rmtree("results")
    os.symlink(drive_results, "results")
    print("results/ ->", drive_results)

In [ ]:
!pip install -q -r requirements.txt
!python -c "from src.data import load_fer2013, check_counts; check_counts(load_fer2013(splits=('train', 'val')))"

## 1. Chạy thử (vài giây) rồi BASE đầy đủ

Chạy thử ghi vào `results/_thu/`, không ảnh hưởng kết quả thật.

In [ ]:
!python -m src.train --run BASE --thu

In [ ]:
!python -m src.train --run BASE --change "baseline"

## 2. Các nhóm thực nghiệm

Run đã có log sẽ được bỏ qua, nên chạy lại ô sau khi bị ngắt là tiếp tục từ run còn thiếu. Kiểm nhanh config bằng `--thu` nếu cần: `!python -m experiments.run_E3 --thu`.

Kaggle T4 x2: ô dưới chạy 2 luồng song song (GPU 0: E1 → E2 → E3, GPU 1: E4 → E5 → E6). E2 tự lấy kiến trúc tốt nhất từ `results/E1/summary.csv` nên nằm sau E1 trong cùng luồng. Chạy riêng một nhóm: `!python -m experiments.run_E4`.

In [ ]:
# Chạy E1–E6. Có 2 GPU (Kaggle T4 x2) thì chia 2 luồng song song: GPU 0 làm E1 -> E2 -> E3
# (E2 cần kết quả E1), GPU 1 làm E4 -> E5 -> E6. Có 1 GPU (Colab) thì chạy tuần tự.
# Log từng luồng ở WORK/log_gpu<i>.txt; cứ 2 phút in dòng cuối để theo dõi.
import torch

streams = [["E1", "E2", "E3"], ["E4", "E5", "E6"]]
if torch.cuda.device_count() < 2:
    streams = [streams[0] + streams[1]]

procs, logs = [], []
for i, groups in enumerate(streams):
    cmd = "; ".join(f"python -m experiments.run_{g}" for g in groups)
    env = dict(os.environ, CUDA_VISIBLE_DEVICES=str(i), PYTHONIOENCODING="utf-8", PYTHONUNBUFFERED="1")
    logs.append(os.path.join(WORK, f"log_gpu{i}.txt"))
    procs.append(subprocess.Popen(["bash", "-c", cmd], env=env, stdout=open(logs[-1], "w"),
                                  stderr=subprocess.STDOUT))
    print(f"GPU {i}: {' -> '.join(groups)}  (log: {logs[-1]})")

def last_line(path):
    lines = [l for l in open(path, encoding="utf-8", errors="replace").read().splitlines() if l.strip()]
    return lines[-1] if lines else "(chưa có output)"

start = time.time()
while any(p.poll() is None for p in procs):
    time.sleep(120)
    print(f"--- {int(time.time() - start) // 60} phút")
    for i, path in enumerate(logs):
        print(f"GPU {i}: {last_line(path)}")

for i, path in enumerate(logs):
    print()
    print(f"===== GPU {i} (mã thoát {procs[i].returncode}) =====")
    print(open(path, encoding="utf-8", errors="replace").read()[-4000:])

## 3. Mô hình cuối (chạy sau khi chốt `configs/final.yaml`)

Ô này tự dừng nếu `configs/final.yaml` còn `chot: false` (không train, không đọc PrivateTest), nên Save & Run All lúc chưa chốt vẫn an toàn.

Trước khi chạy mô hình cuối: commit + push `results/BASE`, `results/E1` … `results/E6` (log, summary) và `configs/final.yaml` đã chốt. Khi đó các ô BASE và E1–E6 ở trên tự bỏ qua run đã có log, chỉ ô này chạy.

Có 2 GPU: GPU 0 train F-0 và F-2, GPU 1 train F-1 song song; xong cả 3 mới chọn seed theo val_loss và đánh giá PrivateTest **một lần**, rồi vẽ filter / feature map / Grad-CAM.

In [ ]:
# Mô hình cuối: train 3 seed (chia 2 GPU nếu có), rồi đánh giá PrivateTest MỘT lần và vẽ hình.
import torch

env = dict(os.environ, PYTHONIOENCODING="utf-8", PYTHONUNBUFFERED="1")
if torch.cuda.device_count() >= 2:
    jobs = []
    for i, seeds in enumerate([["0", "2"], ["1"]]):
        log = os.path.join(WORK, f"log_final_gpu{i}.txt")
        jobs.append((log, subprocess.Popen(
            ["python", "-m", "experiments.run_final", "--seeds", *seeds, "--chi-train"],
            env=dict(env, CUDA_VISIBLE_DEVICES=str(i)), stdout=open(log, "w"), stderr=subprocess.STDOUT)))
    for log, job in jobs:
        job.wait()
        print(f"===== {log} (mã thoát {job.returncode}) =====")
        print(open(log, encoding="utf-8", errors="replace").read()[-3000:])

# Seed nào đã train ở trên thì được bỏ qua; còn lại là đánh giá PrivateTest một lần + vẽ hình.
result = subprocess.run(["python", "-m", "experiments.run_final"], env=env,
                        capture_output=True, text=True, encoding="utf-8", errors="replace")
print(result.stdout[-6000:])
print(result.stderr[-3000:])

## 4. Đóng gói kết quả để tải về

Không kèm `checkpoints/` (nặng, chỉ cần checkpoint mô hình cuối) và `_thu/`.

In [ ]:
name = f"results_{time.strftime('%Y%m%d-%H%M')}.zip"
zip_path = os.path.join(WORK, name)
subprocess.run(["zip", "-qr", zip_path, "results/", "-x", "results/_thu/*", "*/checkpoints/*"], check=True)
print("Đã tạo", zip_path, os.path.getsize(zip_path) // 1024, "KB")
if COLAB:
    from google.colab import files
    files.download(zip_path)